In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

In [23]:
from sklearn import set_config

set_config(display="text")

In [2]:
# --------------------------------------------------
# 1. Load data
# --------------------------------------------------

train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")

TARGET = "label"

X = train.drop(columns=[TARGET])
y = train[TARGET]

X_test = test.drop(columns=[TARGET])
y_test = test[TARGET]

In [4]:
# --------------------------------------------------
# 2. Identify data types
# --------------------------------------------------

numeric_cols = X.select_dtypes(include=np.number).columns.tolist()

categorical_cols = X.select_dtypes(
    include=["object", "string", "category", "bool"]
).columns.tolist()

In [7]:
# --------------------------------------------------
# 3. Preprocessing
# --------------------------------------------------

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_cols),
    ("cat", categorical_transformer, categorical_cols)
])

In [8]:
# --------------------------------------------------
# 4. Cross-validation
# --------------------------------------------------

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [10]:
# 查看标签缺失数量
print(y.isna().sum())

3


In [14]:
print(X.isna().sum())

geological_era       2
weather_pattern      3
region               4
aptitude_score       4
performance_score    2
stability_index      2
personal_interest    6
load_ratio           2
brew_preference      7
instrument           3
species              3
index_weight         3
mineral_type         3
activity_duration    2
composite_rank       1
dtype: int64


In [11]:
# 把X和y一起过滤掉y为空的行
mask = y.notna()
X = X[mask].copy()
y = y[mask].copy()

In [17]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import make_scorer, f1_score

# Check target
print("Target values:")
print(y.value_counts())

# Cross-validation
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# Explicitly define "yes" as the positive class
f1_yes = make_scorer(
    f1_score,
    pos_label="yes"
)

# Logistic Regression pipeline
logistic_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(
        max_iter=2000,
        random_state=42
    ))
])

# Hyperparameter grid
logistic_params = {
    "classifier__C": [0.01, 0.1, 1, 10, 100]
}

# Grid search
logistic_grid = GridSearchCV(
    estimator=logistic_pipeline,
    param_grid=logistic_params,
    cv=cv,
    scoring=f1_yes,
    n_jobs=-1,
    error_score="raise"
)

# Train
logistic_grid.fit(X, y)

print("Best parameters:")
print(logistic_grid.best_params_)

print("Best CV F1:")
print(logistic_grid.best_score_)

Target values:
label
no     23645
yes     7464
Name: count, dtype: int64
Best parameters:
{'classifier__C': 100}
Best CV F1:
0.6260812924570762


In [21]:
# whether the categorical variables are the main reason the Random Forest is taking a long time
preprocessor_check = preprocessor.fit(X)

feature_names = preprocessor_check.get_feature_names_out()

print("Original features:", X.shape[1])
print("After preprocessing:", len(feature_names))

Original features: 15
After preprocessing: 82


In [24]:
# --------------------------------------------------
# 6. Random Forest
# --------------------------------------------------

rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        random_state=42,
        n_jobs=1
    ))
])

rf_params = {
    "classifier__n_estimators": [100, 200],
    "classifier__max_depth": [None, 10],
    "classifier__min_samples_split": [2, 5],
    "classifier__min_samples_leaf": [1, 2]
}

rf_grid = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=rf_params,
    cv=cv,
    scoring=f1_yes,
    n_jobs=-1,
    error_score="raise",
    verbose=2
)

rf_grid.fit(X, y)

Fitting 5 folds for each of 16 candidates, totalling 80 fits


GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
             error_score='raise',
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('num',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(strategy='median')),
                                                                                         ('scaler',
                                                                                          StandardScaler())]),
                                                                         ['aptitude_score',
                                                                          'performance_score',
                                                                          'stability_index',
                                     

In [25]:
# result of random forest
print("Best parameters:")
print(rf_grid.best_params_)

print("Best CV F1:")
print(rf_grid.best_score_)

Best parameters:
{'classifier__max_depth': None, 'classifier__min_samples_leaf': 2, 'classifier__min_samples_split': 2, 'classifier__n_estimators': 100}
Best CV F1:
0.6254519125319463


In [28]:
# --------------------------------------------------
# 7. Test predictions
# --------------------------------------------------

mask = y_test.notna()
X_test = X_test[mask].copy()
y_test = y_test[mask].copy()

models = {
    "Logistic Regression": logistic_grid.best_estimator_,
    "Random Forest": rf_grid.best_estimator_
}

results = []

for name, model in models.items():

    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)[:, 1]

    result = {
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(
            y_test, predictions, pos_label="yes"
        ),
        "Recall": recall_score(
            y_test, predictions, pos_label="yes"
        ),
        "F1": f1_score(
            y_test, predictions, pos_label="yes"
        ),
        "Balanced Accuracy": balanced_accuracy_score(
            y_test, predictions
        ),
        "ROC-AUC": roc_auc_score(
            (y_test == "yes").astype(int),
            probabilities
        )
    }

    results.append(result)

    print("\n==============================")
    print(name)
    print("==============================")

    print(classification_report(y_test, predictions))
    print(confusion_matrix(y_test, predictions))

results_df = pd.DataFrame(results)

print("\nFinal comparison:")
print(results_df)

print("\nBest Logistic Regression parameters:")
print(logistic_grid.best_params_)

print("\nBest Random Forest parameters:")
print(rf_grid.best_params_)


Logistic Regression
              precision    recall  f1-score   support

          no       0.87      0.92      0.90     10165
         yes       0.70      0.57      0.63      3168

    accuracy                           0.84     13333
   macro avg       0.79      0.75      0.76     13333
weighted avg       0.83      0.84      0.83     13333

[[9386  779]
 [1357 1811]]

Random Forest
              precision    recall  f1-score   support

          no       0.87      0.92      0.90     10165
         yes       0.69      0.57      0.62      3168

    accuracy                           0.84     13333
   macro avg       0.78      0.74      0.76     13333
weighted avg       0.83      0.84      0.83     13333

[[9354  811]
 [1369 1799]]

Final comparison:
                 Model  Accuracy  Precision    Recall        F1  \
0  Logistic Regression  0.839796   0.699228  0.571654  0.629038   
1        Random Forest  0.836496   0.689272  0.567866  0.622707   

   Balanced Accuracy   ROC-AUC  
0 